### Convolutional Neural Networks (CNN)

The third major neural architecture family alongside RNN/LSTM (`nlp.ipynb`) and Transformers (`llm-mechanics/llm-architecture.ipynb`). Less directly relevant to text/tabular fraud work, included for completeness. Covers the convolution operation, why it beats a fully-connected layer for grid-structured data, padding/stride, pooling, and channels.

#### 0. The convolution operation, worked by hand

A small kernel (filter) slides over the input, at each position computing the element-wise product with the patch underneath it, summed into one output value.

Toy setup: 3x3 input, 2x2 kernel, stride 1, no padding.
```
input:            kernel:
1 2 1             1 0
0 1 3             0 1
2 0 1
```
Output size = (3-2)/1 + 1 = 2x2. Each output cell = element-wise product of the kernel with the input patch underneath it, summed:
```
output[0,0] = input[0:2,0:2] . kernel = 1*1 + 2*0 + 0*0 + 1*1 = 1+0+0+1 = 2
output[0,1] = input[0:2,1:3] . kernel = 2*1 + 1*0 + 1*0 + 3*1 = 2+0+0+3 = 5
output[1,0] = input[1:3,0:2] . kernel = 0*1 + 1*0 + 2*0 + 0*1 = 0+0+0+0 = 0
output[1,1] = input[1:3,1:3] . kernel = 1*1 + 3*0 + 0*0 + 1*1 = 1+0+0+1 = 2

output:
2 5
0 2
```
This particular kernel (1 in the top-left and bottom-right, 0 elsewhere) sums the two diagonal corners of each patch, a toy "filter," a real trained kernel might instead learn to respond strongly to a specific pattern, an edge, a corner, a texture, wherever that pattern appears in the input.

In [ ]:
import numpy as np

input_grid = np.array([[1, 2, 1], [0, 1, 3], [2, 0, 1]])
kernel = np.array([[1, 0], [0, 1]])

def convolve2d(x, k):
    kh, kw = k.shape
    oh, ow = x.shape[0] - kh + 1, x.shape[1] - kw + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(x[i:i+kh, j:j+kw] * k)
    return out

print("output:\n", convolve2d(input_grid, kernel))

#### 1. Why convolution instead of a fully-connected layer

Parameter sharing: the SAME kernel (4 numbers in the toy example) is reused at every position across the whole input. A fully-connected layer mapping a 3x3 input to a 2x2 output would need 9*4=36 separate weights, no sharing at all, every output unit has its own independent connection to every input unit. Convolution needs only the kernel's own parameters (4 here), regardless of how large the input grid is, a dramatic parameter reduction that scales to real images (millions of pixels) without exploding parameter count.

Translation invariance: because the same kernel scans the whole input, a feature detector that fires on a pattern in one location fires on that same pattern anywhere else in the input too, without needing to separately learn it at every position. A fully-connected layer has no such built-in property, a pattern learned in the top-left corner tells it nothing about the same pattern appearing in the bottom-right.

#### 2. Padding and stride

Padding: add a border of zeros around the input before convolving, so the output does not shrink every layer (without padding, a 3x3 input with a 2x2 kernel always shrinks to 2x2, stack enough conv layers and the spatial size eventually hits zero). "Same" padding keeps the output size equal to the input size.

Stride: how many positions the kernel moves per step. Stride 1 (used above) moves one position at a time, the most common default. Stride 2 skips every other position, halving the output size roughly, cheaper computation, less spatial detail kept.

Output size formula: `output = floor((input + 2*padding - kernel) / stride) + 1`. Worked check on the toy example (input=3, padding=0, kernel=2, stride=1): floor((3+0-2)/1)+1 = floor(1)+1 = 2, matches the 2x2 output computed above.

#### 3. Pooling

Max pooling: slide a window over the feature map, keep only the MAXIMUM value in each window, discard the rest. Reduces spatial dimensions (cheaper for later layers) and adds a small amount of translation invariance (a feature shifted by 1 pixel still likely falls in the same pooling window, giving the identical max).

Worked example, 2x2 max pooling with stride 2 on the convolution output above:
```
2 5        max pooling with a 2x2 window over the whole 2x2 output
0 2   ->   max(2,5,0,2) = 5
```
The whole 2x2 feature map collapses to a single value, 5, the strongest signal detected anywhere in that region survives, the rest is discarded.

In [ ]:
import torch
import torch.nn as nn

x = torch.tensor([[[[1., 2., 1.], [0., 1., 3.], [2., 0., 1.]]]])  # shape (batch=1, channels=1, 3, 3)

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)
conv.weight.data = torch.tensor([[[[1., 0.], [0., 1.]]]])  # set to the same toy kernel

conv_out = conv(x)
print("conv output:\n", conv_out)

pool = nn.MaxPool2d(kernel_size=2)
print("max pooled:\n", pool(conv_out))

#### 4. Channels and stacking layers

A real input has multiple channels (RGB = 3 channels for a color image), and each conv layer typically learns MANY kernels in parallel (not just 1), each producing its own output feature map, these stack together as the channels for the next layer. Early layers tend to learn simple features (edges, colors), later layers combine those into increasingly complex, abstract features (shapes, textures, eventually object parts), each layer's kernels operate on the PREVIOUS layer's already-detected features, not the raw pixels directly, the same "features feed into features" composability idea as a deep network in general.